In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
pd.options.display.max_rows = 999
import warnings
warnings.filterwarnings("ignore")
import sys
sys.dont_write_bytecode = True
#importing datasets
from src.New_Utils import Sequences_Night
from Datasets import Replace
from Datasets import AIDE
from Datasets import Shanghai
#Libraries to run the experiments
from src.model import Model
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Codigo Noche

## Secuencias de noche

In [ ]:
sequences = Sequences_Night(Replace)
sequences_noche = [seq for seq in sequences if seq['Y_total'] == 1]

## Modelo noche

In [ ]:
model_night = Model()
results_train = []
results_test = []

Data_ROC_train = []
Data_ROC_test = []

l1_penalty=0.005
l2_penalty=0.05
learning_rate=0.00001
batch_size=32

In [ ]:
# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
        # obtaining the folds to use
    train_fold = [sequences[idx] for idx in train_index]
    test_fold = [sequences[idx] for idx in test_index]
    
    # fit the model to the training set
    model_night.fit(
        sequences=train_fold,
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=l1_penalty,
        l2_penalty=l2_penalty,
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=1000,
        seed=42,
        verbose=0
    )
    # predicting the values on the fold's data
    train_probs = np.array(model_night.predict(sequences=train_fold)).flatten()
    test_probs = np.array(model_night.predict(sequences=test_fold)).flatten()
    # Create a new list for holding the sequences, the mark of the sequence and the predicted probability
    for idx, seq in enumerate(train_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = train_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_train.append(new_seq)

    # 4. Procesamos el Test
    for idx, seq in enumerate(test_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = test_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_test.append(new_seq)
        # evaluate the model on the traiing and testing sets
        metrics_train = model_night.evaluate(sequences=train_fold)
        metrics_test = model_night.evaluate(sequences=test_fold)
    # save the results
    results_train.append(metrics_train)
    results_test.append(metrics_test)
    

In [ ]:
# este es optimizado
import os
import psutil
import numpy as np
from joblib import Parallel, delayed

def process_fold(i, train_index, test_index, sequences, params):
    # --- MONITOREO DE RAM INICIAL ---
    process = psutil.Process(os.getpid())
    mem_inicio = process.memory_info().rss / (1024 ** 2) # Convertir a MB
    
    # 1. Obtención de los folds
    train_fold = [sequences[idx] for idx in train_index]
    test_fold = [sequences[idx] for idx in test_index]
    
    # 2. Configuración y Entrenamiento
    # Nota: Asegúrate de que model_night sea una instancia fresca o reseteada
    local_model = model_night 
    
    local_model.fit(
        sequences=train_fold,
        sequence_length=params['seq_len'],
        l1_penalty=params['l1'],
        l2_penalty=params['l2'],
        learning_rate=params['lr'],
        batch_size=params['bs'],
        epochs=1000,
        seed=42,
        verbose=0
    )
    
    # 3. Predicciones (Fuera de bucles innecesarios)
    train_probs = np.array(local_model.predict(sequences=train_fold)).flatten()
    test_probs = np.array(local_model.predict(sequences=test_fold)).flatten()
    
    # 4. Evaluación de métricas (UNA SOLA VEZ por fold)
    # Esto corrige el error del bucle que ralentizaba todo
    m_train = local_model.evaluate(sequences=train_fold)
    m_test = local_model.evaluate(sequences=test_fold)
    
    # 5. Preparación de datos para ROC
    fold_train_data = []
    for idx, seq in enumerate(train_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = train_probs[idx]
        new_seq['fold'] = i + 1
        fold_train_data.append(new_seq)

    fold_test_data = []
    for idx, seq in enumerate(test_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = test_probs[idx]
        new_seq['fold'] = i + 1
        fold_test_data.append(new_seq)

    # --- MONITOREO DE RAM FINAL ---
    mem_final = process.memory_info().rss / (1024 ** 2)
    print(f"✅ Fold {i+1} completo | RAM usada: {mem_final - mem_inicio:.2f} MB | Total proceso: {mem_final:.2f} MB")
    
    return fold_train_data, fold_test_data, m_train, m_test

# --- CONFIGURACIÓN DE EJECUCIÓN ---

# Definimos n_jobs (con 24GB de RAM, puedes probar con 8 o 12)
N_JOBS = 8 

params_config = {
    'seq_len': int(7 * 24 * 60 // 5),
    'l1': l1_penalty, 
    'l2': l2_penalty,
    'lr': learning_rate, 
    'bs': batch_size
}

# Ejecución paralela
resultados_paralelos = Parallel(n_jobs=N_JOBS)(
    delayed(process_fold)(i, t_idx, v_idx, sequences, params_config) 
    for i, (t_idx, v_idx) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences]))
)

# Unificar resultados
for f_train, f_test, m_train, m_test in resultados_paralelos:
    Data_ROC_train.extend(f_train)
    Data_ROC_test.extend(f_test)
    results_train.append(m_train)
    results_test.append(m_test)